# Inductive biases for transitive inference
Now that we have a geometric solution for transitive inference in `geometric_solution.ipynb`, how can we bias our networks to learn this solution? In this notebook, we'll bias them to learn stimulus embeddings that lie on the unit circle. In effect, if the only parameter the network can tune is the angle of the embedding, then perhaps the network is biased towards learning a solution that generalizes.

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from relational_attention.tasks import TransitiveInferenceDataset

In [2]:
torch.manual_seed(42)

## Build task

In [3]:
# Parameters
num_samples = 1000
num_items = 5

In [4]:
# Training task
dataset_train = TransitiveInferenceDataset(num_samples=num_samples, num_items=num_items)
loader_train = DataLoader(dataset_train, batch_size=16, shuffle=True)

In [5]:
# Testing task
dataset_test = TransitiveInferenceDataset(num_samples=num_samples, num_items=num_items, train=False)
loader_test = DataLoader(dataset_test, batch_size=16, shuffle=True)

## Build models

### Minimal model

In [6]:
class RelationalAttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        # Initialize weights
        self.W_q = nn.Linear(embed_dim, head_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, head_dim, bias=False)

    def forward(self, x_1, x_2):
        """
        x_1: (batch, embed_dim)
        x_2: (batch, embed_dim)
        returns: (batch, seq_len, head_dim)
        """
        Q = self.W_q(x_2)                            # (B, D)
        K = self.W_k(x_1)                            # (B, D)

        # Compute attention scores over the *learned* Value slots
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (K.size(-1) ** 0.5)  # (B, L, L)
        return attn_weights.flatten()

### Normalized model

In [7]:
class UnitNormLinear(nn.Module):
    def __init__(self, in_features, out_features, dim_to_normalize=0, bias=False, eps=1e-8):
        super().__init__()
        self.weight_raw = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.dim = dim_to_normalize  # 0 = rows, 1 = columns
        self.eps = eps

    def forward(self, x):
        # weight_raw shape: (out_features, in_features)
        # Normalize along chosen dim
        w = self.weight_raw
        norm = w.norm(dim=self.dim, keepdim=True).clamp_min(self.eps)
        w_hat = w / norm

        # Standard linear: x @ W^T
        out = x @ w_hat.t()
        if self.bias is not None:
            out = out + self.bias
        return out

In [8]:
class RelationalNormAttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        # Initialize weights
        self.W_q = UnitNormLinear(embed_dim, head_dim, dim_to_normalize=1, bias=False)
        self.W_k = UnitNormLinear(embed_dim, head_dim, dim_to_normalize=1, bias=False)

    def forward(self, x_1, x_2):
        """
        x_1: (batch, embed_dim)
        x_2: (batch, embed_dim)
        returns: (batch, seq_len, head_dim)
        """
        Q = self.W_q(x_2)                            # (B, D)
        K = self.W_k(x_1)                            # (B, D)

        # Compute attention scores over the *learned* Value slots
        attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (K.size(-1) ** 0.5)  # (B, L, L)
        return attn_weights.flatten()

### Norm penalty model

In [9]:
def unit_norm_penalty(W, dim=1):
    # W shape: (out, in) – encourage columns to be unit norm if dim=0, rows if dim=1
    norms = W.norm(dim=dim)
    return ((norms - 1.0) ** 2).mean()

In [10]:
def train_model_penalty(
    model,
    data_loader,
    lambda_u=1e-3,
    penalty=False,
    optimizer=None,
    loss_fn=None,
    epochs=10,
    device=None,
    return_history=True,
    verbose=True,
):
    """
    Generic training loop with tqdm progress bar.

    Args:
        model (nn.Module): PyTorch model to train.
        data_loader (DataLoader): Iterable providing (X, y) batches.
        optimizer (torch.optim.Optimizer, optional): Optimizer instance.
            If None, defaults to Adam(model.parameters()).
        loss_fn (callable, optional): Loss function.
            If None, defaults to BCEWithLogitsLoss.
        epochs (int): Number of training epochs.
        device (torch.device, optional): Device for computation.
            If None, inferred from model or defaults to CPU.
        return_history (bool): Whether to return the list of losses per epoch.
        verbose (bool): Whether to show tqdm progress bar.

    Returns:
        list[float]: Average loss per epoch (if return_history=True).
    """
    # Defaults
    device = device or next(model.parameters()).device
    optimizer = optimizer or torch.optim.Adam(model.parameters())
    loss_fn = loss_fn or torch.nn.BCEWithLogitsLoss()

    # Put model in training mode
    model.train()

    loss_history = []

    # Optionally wrap tqdm
    progress = range(epochs)
    if verbose:
        progress = tqdm(progress, desc="Training", total=epochs, leave=True)

    for epoch in progress:
        total_loss = 0.0

        for X, y in data_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()

            preds = model(X[:,0:1,:], X[:,1:2,:])
            loss_main = loss_fn(preds, y)

            if penalty:
                penalty = unit_norm_penalty(model.W_q.weight, dim=0) \
                        + unit_norm_penalty(model.W_k.weight, dim=0)
                loss = loss_main + lambda_u * penalty
            else:
                loss = loss_main
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(data_loader)
        loss_history.append(avg_loss)

        # Update progress bar if active
        if verbose:
            progress.set_postfix({"loss": f"{avg_loss:.4f}"})

    if verbose:
        progress.close()

    return loss_history if return_history else None

## Initialize models

In [11]:
# Base model with no changes
model_base = RelationalAttentionHead(num_items, 2)

In [12]:
# Model constrained to unit vector weights
model_norm = RelationalNormAttentionHead(num_items, 2)

In [13]:
# Base model with penalty on weight norms
model_penalty = RelationalAttentionHead(num_items, 2)

## Train models

In [14]:
lr = 1e-3
loss_fn = torch.nn.BCEWithLogitsLoss()
epochs = 250

In [15]:
optimizer = torch.optim.Adam(model_base.parameters(), lr=lr)
loss_base = train_model_penalty(model_base, loader_train, optimizer=optimizer, loss_fn=loss_fn, epochs=epochs,)

Training: 100%|██████████| 250/250 [00:04<00:00, 59.36it/s, loss=0.0000]


In [16]:
optimizer = torch.optim.Adam(model_norm.parameters(), lr=lr)
loss_norm = train_model_penalty(model_norm, loader_train, optimizer=optimizer, loss_fn=loss_fn, epochs=epochs,)

Training: 100%|██████████| 250/250 [00:05<00:00, 49.00it/s, loss=0.5523]


In [17]:
optimizer = torch.optim.Adam(model_penalty.parameters(), lr=lr)
loss_penalty = train_model_penalty(model_penalty, loader_train, penalty=True, optimizer=optimizer, loss_fn=loss_fn, epochs=epochs,)

Training: 100%|██████████| 250/250 [00:06<00:00, 41.24it/s, loss=0.0094]


## Evaluate models

In [18]:
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            logits = model(X[:,0:1,:], X[:,1:2,:])
            preds = (torch.sigmoid(logits) > 0.5)

            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [19]:
acc_base = evaluate_model(model_base, loader_test)
acc_norm = evaluate_model(model_norm, loader_test)
acc_penalty = evaluate_model(model_penalty, loader_test)

print("Base model:", acc_base)
print("Enforced norm weights:", acc_norm)
print("Norm penalty:", acc_penalty)

Base model: 0.327
Enforced norm weights: 0.328
Norm penalty: 0.343
